# GPU Memory Hierarchy & Data Movement with CuTe DSL

A modern NVIDIA GPU has **5 levels of memory**, each trading off size for speed:

```
                    Scope         Capacity (typical)    Latency        Bandwidth
  +-----------+
  |   RMEM    |     Per-thread    255 registers/thread  ~0 cycles      ~unlimited
  +-----------+
  | L1 Cache  |     Per-SM        128-256 KB            ~30 cycles     hardware-managed
  +-----------+
  |   SMEM    |     Per-block     up to 228 KB (Ada)    ~20-30 cycles  ~19 TB/s (aggregate)
  +-----------+
  | L2 Cache  |     Device-wide   up to 96 MB (Ada)     ~200 cycles    hardware-managed
  +-----------+
  |   GMEM    |     Device-wide   up to 80 GB (A100)    ~400 cycles    ~2 TB/s
  +-----------+
```

**Three of these are programmer-controlled:** Global Memory (GMEM), Shared Memory (SMEM), and Register Memory (RMEM). The L1 and L2 caches are hardware-managed data flows through them automatically, though we can influence their behavior with cache hints.

The fundamental job of a high-performance GPU kernel is to **move data up this hierarchy** as efficiently as possible: from GMEM into SMEM (shared across a thread block), then from SMEM into RMEM (private to each thread), where the actual computation happens.

CuTe DSL provides **CopyAtoms**, abstractions over the hardware copy instructions to express these data movement patterns. This notebook explores each memory level and demonstrates how to move data between them.

**Authors: Claude Code & Pramodith**

In [11]:
%pip install -q torch triton nvidia-cutlass nvidia-cutlass-dsl

/workspace/kernel-engineering/.venv/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.


In [12]:
import os
import torch
import triton

# Must be set BEFORE importing cutlass -- the EnvManager reads env vars at import time
os.environ["CUTE_DSL_KEEP_PTX"] = "1"

import cutlass
import cutlass.cute as cute
from cutlass.cute.runtime import from_dlpack

assert torch.cuda.is_available(), "CUDA GPU required"
major, minor = torch.cuda.get_device_capability()
os.environ["CUTE_DSL_ARCH"] = f"sm_{major}{minor}" + ("a" if major >= 9 else "")

print(f"GPU: {torch.cuda.get_device_name()}")
print(f"Compute capability: sm_{major}{minor}")
print(f"Target arch: {os.environ['CUTE_DSL_ARCH']}")

GPU: NVIDIA GeForce RTX 5060
Compute capability: sm_120
Target arch: sm_120a


## The 5 Memory Levels in Detail

### 1. Global Memory (GMEM)
- **Scope:** Visible to all threads on the device
- **Backed by:** HBM (High Bandwidth Memory) or GDDR
- **Size:** GBs (e.g., 12 GB on RTX 4070, 80 GB on A100)
- **Latency:** ~400-800 cycles
- **Access pattern:** Coalesced 128-byte transactions. When 32 threads in a warp access consecutive addresses, the hardware merges them into minimal transactions. Scattered access wastes bandwidth.

This is where your PyTorch tensors live. Every kernel starts and ends by reading from and writing to GMEM.

### 2. L2 Cache
- **Scope:** Device-wide (shared by all SMs)
- **Size:** MBs (e.g., 36 MB on RTX 4070, 40 MB on A100, 50 MB on H100)
- **Latency:** ~200 cycles
- **Managed by:** Hardware (transparent caching of GMEM accesses)
- **Programmer influence:** Cache eviction hints (`EVICT_FIRST`, `EVICT_LAST`, `EVICT_NORMAL`) via CopyAtom parameters, and L2 persistence controls via `cudaAccessPolicyWindow`

### 3. Shared Memory (SMEM)
- **Scope:** Per-thread-block (all threads in a block see the same SMEM)
- **Size:** Configurable per-block, up to 228 KB on Ampere/Hopper
- **Latency:** ~20-30 cycles
- **Key feature:** Software-managed scratchpad. The programmer explicitly allocates and fills it.
- **Why it matters:** When multiple threads in a block need the same data, loading it once into SMEM and reading it N times is far cheaper than N separate GMEM loads.

### 4. L1 Cache
- **Scope:** Per-SM
- **Size:** 128-256 KB (shared/configurable with SMEM on some architectures)
- **Latency:** ~30 cycles
- **Managed by:** Hardware (caches GMEM and register spills)
- **Note:** On modern GPUs (Ampere+), L1 and SMEM share the same on-chip SRAM. The split is configurable via `cudaFuncSetAttribute`.

### 5. Register Memory (RMEM)
- **Scope:** Per-thread (completely private)
- **Size:** Up to 255 32-bit registers per thread
- **Latency:** ~0 cycles (operands are directly available to ALUs)
- **Key feature:** This is where computation actually happens. Instructions like FMA read their operands from registers and write results back to registers.
- **Tradeoff:** More registers per thread = fewer threads per SM (lower occupancy). The compiler manages register allocation, but CuTe DSL's `make_rmem_tensor` lets you explicitly allocate register-backed tensors.

## Data Movement Paths

Not all memory-to-memory paths are equal. The GPU hardware provides specialized instructions for certain paths:

```
  GMEM ──────────────────────────────────> RMEM     (LD.GLOBAL, load through L1/L2)
  GMEM ──────────────────> SMEM                     (CP.ASYNC, bypasses registers!)
                           SMEM ────────> RMEM      (LDS, load from shared memory)
                           SMEM ────────> RMEM      (LDMATRIX, warp-level structured load)
  GMEM <──────────────────────────────── RMEM       (ST.GLOBAL, store through L1/L2)
                           SMEM <──────── RMEM      (STS, store to shared memory)
```

The key insight is that **GMEM → SMEM can bypass registers entirely** using `cp.async` (Ampere+). This is important because registers are a scarce resource -- using them as a waypoint for data that's just passing through to SMEM is wasteful.

In CuTe DSL, each of these hardware paths is represented by a **CopyAtom**, a type that encapsulates the instruction, its operand layout, and how threads cooperate to move data. Let's explore each path.

## Path 1: GMEM → RMEM → GMEM (Direct Load/Store)

The simplest data movement pattern: each thread loads data from global memory directly into its registers, computes on it, and stores results back. This is what happens when you index into a GMEM tensor inside a kernel.

Under the hood, the GPU issues `LD.GLOBAL` instructions that travel through L2 → L1 → registers. We don't need to explicitly manage SMEM at all.

Let's write a simple kernel that loads elements from GMEM into register-backed tensors using `make_rmem_tensor`, doubles them, and writes them back.

In [13]:
ELEMS_PER_THREAD = 4

@cute.kernel
def gmem_to_rmem_kernel(
    gIn: cute.Tensor,
    gOut: cute.Tensor,
):
    """Each thread loads ELEMS_PER_THREAD elements from GMEM into registers,
    doubles them, and stores back to GMEM."""
    tidx, _, _ = cute.arch.thread_idx()
    bidx, _, _ = cute.arch.block_idx()
    bdim, _, _ = cute.arch.block_dim()

    global_tid = bidx * bdim + tidx

    # Allocate a register-backed tensor (RMEM) to hold this thread's data
    rmem = cute.make_rmem_tensor(ELEMS_PER_THREAD, cutlass.Float32)

    # GMEM → RMEM: load elements into registers
    for i in range(ELEMS_PER_THREAD):
        rmem[i] = gIn[global_tid * ELEMS_PER_THREAD + i]

    # Compute in registers (doubling each element)
    for i in range(ELEMS_PER_THREAD):
        rmem[i] = rmem[i] * 2.0

    # RMEM → GMEM: store results back
    for i in range(ELEMS_PER_THREAD):
        gOut[global_tid * ELEMS_PER_THREAD + i] = rmem[i]


@cute.jit
def gmem_to_rmem(mIn: cute.Tensor, mOut: cute.Tensor):
    N = mIn.shape[0]
    threads_per_block = 256
    blocks = N // (threads_per_block * ELEMS_PER_THREAD)
    gmem_to_rmem_kernel(mIn, mOut).launch(
        grid=(blocks, 1, 1),
        block=(threads_per_block, 1, 1),
    )


# Test correctness
N = 1 << 20  # ~1M elements
inp = torch.randn(N, device="cuda", dtype=torch.float32)
out = torch.zeros(N, device="cuda", dtype=torch.float32)

inp_ = from_dlpack(inp, assumed_align=16)
out_ = from_dlpack(out, assumed_align=16)

gmem_to_rmem_fn = cute.compile(gmem_to_rmem, inp_, out_)
gmem_to_rmem_fn(inp_, out_)

torch.testing.assert_close(out, inp * 2.0)
print(f"GMEM → RMEM → GMEM: PASSED (N={N:,})")
print(f"  Input sample:  {inp[:4].tolist()}")
print(f"  Output sample: {out[:4].tolist()}")
print(f"  PTX:\n{gmem_to_rmem_fn.__ptx__}")

GMEM → RMEM → GMEM: PASSED (N=1,048,576)
  Input sample:  [0.4033033847808838, -0.5069962739944458, -0.16861256957054138, -0.6867151260375977]
  Output sample: [0.8066067695617676, -1.0139925479888916, -0.33722513914108276, -1.3734302520751953]
  PTX:
//
// Generated by NVIDIA NVVM Compiler
//
// Compiler Build ID: CL-36006120
// Cuda compilation tools, release 12.9, V12.9.83
// Based on NVVM 20.0.0
//

.version 8.8
.target sm_120a
.address_size 64

	// .globl	kernel_cutlass_gmem_to_rmem_kernel_tensorptrf32gmemalign16o10485761_tensorptrf32gmemalign16o10485761_0

.visible .entry kernel_cutlass_gmem_to_rmem_kernel_tensorptrf32gmemalign16o10485761_tensorptrf32gmemalign16o10485761_0(
	.param .align 8 .b8 kernel_cutlass_gmem_to_rmem_kernel_tensorptrf32gmemalign16o10485761_tensorptrf32gmemalign16o10485761_0_param_0[8],
	.param .align 8 .b8 kernel_cutlass_gmem_to_rmem_kernel_tensorptrf32gmemalign16o10485761_tensorptrf32gmemalign16o10485761_0_param_1[8]
)
.reqntid 256, 1, 1
{
	.reg .b32 	%r<6>

### What happened under the hood

1. **`cute.make_rmem_tensor(4, cutlass.Float32)`** allocated 4 FP32 values in registers. This compiles down to 4 local variables in the PTX, each backed by a 32-bit register.

2. **`rmem[i] = gIn[...]`** generated `LD.GLOBAL` instructions. The data travels: GMEM → L2 → L1 → register file.

3. **`rmem[i] = rmem[i] * 2.0`** is a pure register operation (`FMUL`). Both operands and the result are in registers, zero memory traffic.

4. **`gOut[...] = rmem[i]`** generated `ST.GLOBAL` instructions. Data travels: register file → L1 → L2 → GMEM.

This is the simplest pattern but also the least efficient for kernels where multiple threads need overlapping data -- each thread independently fetches from GMEM, wasting bandwidth on duplicate loads.

_Note: We could've also just done gOut[...] = gIn[...] * 2.0 without using `make_rmem_tensor` at all, and the compiler would still generate the same code. The explicit RMEM tensor just makes it clearer that these values are register-resident and not spilled to memory_.

## Path 2: GMEM → SMEM → RMEM (The Two-Stage Pattern)

When threads within a block need overlapping data, we can save GMEM bandwidth by:
1. **Loading data from GMEM into SMEM** once (cooperatively across all threads in the block)
2. **Having each thread read from SMEM** into its registers

This is the bread-and-butter pattern for GEMM, convolution, reduction, and stencil kernels.

In the kernel below, we demonstrate this explicitly:
- Each thread cooperatively loads part of a block-sized chunk from GMEM into SMEM
- After a `__syncthreads()`, each thread reads from SMEM into registers
- The thread computes on registers and writes results back to GMEM

In [14]:
BLOCK_SIZE = 256
ELEMS_PER_THREAD_SMEM = 4
SMEM_SIZE_NAIVE = BLOCK_SIZE * ELEMS_PER_THREAD_SMEM

@cute.kernel
def gmem_smem_rmem_kernel(
    gIn: cute.Tensor,
    gOut: cute.Tensor,
):
    """Demonstrates the GMEM → SMEM → RMEM → GMEM data movement pattern.
    Each thread block cooperatively loads a chunk into SMEM, then each thread
    reads its elements from SMEM into registers, computes, and writes back."""
    tidx, _, _ = cute.arch.thread_idx()
    bidx, _, _ = cute.arch.block_idx()

    # Step 1: Allocate shared memory for the block (SMEM)
    smem_ptr = cute.arch.alloc_smem(cutlass.Float32, SMEM_SIZE_NAIVE, alignment=16)
    smem = cute.make_tensor(smem_ptr, SMEM_SIZE_NAIVE)

    # Step 2: GMEM → SMEM: each thread loads ELEMS_PER_THREAD_SMEM elements cooperatively
    base_global = bidx * SMEM_SIZE_NAIVE + tidx * ELEMS_PER_THREAD_SMEM
    base_smem = tidx * ELEMS_PER_THREAD_SMEM
    for i in range(ELEMS_PER_THREAD_SMEM):
        smem[base_smem + i] = gIn[base_global + i]

    # Step 3: Synchronize, ensure all threads have finished writing to SMEM
    cute.arch.sync_threads()

    # Step 4: SMEM → RMEM: each thread reads its elements into registers
    rmem = cute.make_rmem_tensor(ELEMS_PER_THREAD_SMEM, cutlass.Float32)
    for i in range(ELEMS_PER_THREAD_SMEM):
        rmem[i] = smem[base_smem + i]

    # Step 5: Compute in registers
    for i in range(ELEMS_PER_THREAD_SMEM):
        rmem[i] = rmem[i] * 2.0

    # Step 6: RMEM → GMEM: write results back
    for i in range(ELEMS_PER_THREAD_SMEM):
        gOut[base_global + i] = rmem[i]


@cute.jit
def gmem_smem_rmem(mIn: cute.Tensor, mOut: cute.Tensor):
    N = mIn.shape[0]
    gmem_smem_rmem_kernel(mIn, mOut).launch(
        grid=(N // SMEM_SIZE_NAIVE, 1, 1),
        block=(BLOCK_SIZE, 1, 1),
    )


# Test correctness
N = 1 << 20
inp = torch.randn(N, device="cuda", dtype=torch.float32)
out = torch.zeros(N, device="cuda", dtype=torch.float32)

inp_ = from_dlpack(inp, assumed_align=16)
out_ = from_dlpack(out, assumed_align=16)

gmem_smem_rmem_fn = cute.compile(gmem_smem_rmem, inp_, out_)
gmem_smem_rmem_fn(inp_, out_)

torch.testing.assert_close(out, inp * 2.0)
print(f"GMEM → SMEM → RMEM → GMEM: PASSED (N={N:,})")
print(f"  Input sample:  {inp[:4].tolist()}")
print(f"  Output sample: {out[:4].tolist()}")

GMEM → SMEM → RMEM → GMEM: PASSED (N=1,048,576)
  Input sample:  [-0.14993590116500854, 0.9357980489730835, 1.6745851039886475, 0.819290816783905]
  Output sample: [-0.2998718023300171, 1.871596097946167, 3.349170207977295, 1.63858163356781]


### Breaking down the SMEM pattern

The key CuTe DSL primitives for shared memory:

| Step | CuTe DSL Call | What it does |
|------|--------------|--------------|
| Allocate SMEM | `cute.arch.alloc_smem(dtype, num_elems, alignment)` | Statically allocates a block of shared memory, returns a pointer |
| Create SMEM tensor | `cute.make_tensor(smem_ptr, shape)` | Wraps the SMEM pointer with a layout to get an indexable tensor |
| GMEM → SMEM | `smem[tidx] = gIn[global_idx]` | Each thread loads one element (travels GMEM → L2 → L1 → RMEM → SMEM) |
| Synchronize | `cute.arch.sync_threads()` | Barrier that ensures all threads have finished their SMEM writes |
| SMEM → RMEM | `rmem[0] = smem[tidx]` | Thread reads from SMEM into a register |

Note that the naive `smem[tidx] = gIn[idx]` path actually goes **GMEM → registers → SMEM** (two hops). The data briefly passes through registers because the basic load/store instructions require register operands. On Ampere+, `cp.async` can bypass this register waypoint -- we'll see that next.

### A note on alignment

Both `from_dlpack(tensor, assumed_align=N)` and `alloc_smem(dtype, n, alignment=N)` take an alignment parameter. **Alignment means the base memory address is guaranteed to be a multiple of N bytes.** This matters because the compiler can emit wider, faster instructions when it knows the alignment:

| Alignment | Widest Instruction | Bytes/op | FP32 elements/op |
|-----------|-------------------|----------|-------------------|
| 4 bytes   | `LD.GLOBAL.32`    | 4        | 1                 |
| 8 bytes   | `LD.GLOBAL.64`    | 8        | 2                 |
| 16 bytes  | `LD.GLOBAL.128`   | 16       | 4                 |

**16 bytes is the sweet spot** -- `LD.GLOBAL.128` is the widest standard load the GPU has, so alignment beyond 16 doesn't help for regular GMEM ↔ RMEM paths. The one exception is `cp.async.bulk` (TMA on Hopper+), which benefits from up to 128-byte alignment for bulk transfers.

PyTorch's CUDA allocator returns 256-byte aligned pointers, so `assumed_align=16` is always safe. For SMEM, `alignment=16` ensures `cp.async` can use its widest transfer mode. If you specify an alignment the data doesn't actually have, wider loads will read garbage or fault.

## Path 3: GMEM → SMEM via `cp.async` (Register-Free Transfer)

Starting with Ampere (SM80), NVIDIA introduced `cp.async` -- an instruction that copies data directly from global memory to shared memory **without using registers as intermediaries**. This has two benefits:

1. **Saves registers:** The data never touches the register file, leaving more registers for computation.
2. **Asynchronous:** The copy is initiated and the thread can continue executing other instructions. The thread only waits when it actually needs the data in SMEM.

In CuTe DSL, this path uses `CopyG2SOp` (Copy Global-to-Shared Operation):

```python
op = cute.nvgpu.cpasync.CopyG2SOp()
atom = cute.make_copy_atom(op, cutlass.Float32, num_bits_per_copy=128)
```

The `num_bits_per_copy` parameter controls the transaction size. 128 bits means 4 FP32 elements per copy operation per thread. After issuing the copy, we call `cute.arch.cp_async_commit_group()` to commit the pending async copies and `cute.arch.cp_async_wait_group(0)` to wait for all pending groups to complete.

In [15]:
BLOCK_SIZE_ASYNC = 256
ELEMS_PER_THREAD_ASYNC = 4
SMEM_SIZE = BLOCK_SIZE_ASYNC * ELEMS_PER_THREAD_ASYNC  # 1024 elements per block

@cute.kernel
def cp_async_kernel(
    gIn: cute.Tensor,
    gOut: cute.Tensor,
):
    """Demonstrates GMEM → SMEM via cp.async, then SMEM → RMEM → GMEM."""
    tidx, _, _ = cute.arch.thread_idx()
    bidx, _, _ = cute.arch.block_idx()

    # Allocate SMEM
    smem_ptr = cute.arch.alloc_smem(cutlass.Float32, SMEM_SIZE, alignment=16)
    smem = cute.make_tensor(smem_ptr, SMEM_SIZE)

    # Build the cp.async copy atom (32-bit per operation)
    # Note: 128-bit cp.async requires statically provable 16-byte alignment,
    # which needs proper tiling infrastructure (covered in a future notebook).
    # Here we use 32-bit granularity to demonstrate the async mechanism.
    cp_async_op = cute.nvgpu.cpasync.CopyG2SOp()
    cp_async_atom = cute.make_copy_atom(cp_async_op, cutlass.Float32, num_bits_per_copy=32)

    # Each thread is responsible for ELEMS_PER_THREAD_ASYNC contiguous elements
    base_idx = bidx * SMEM_SIZE + tidx * ELEMS_PER_THREAD_ASYNC
    smem_offset = tidx * ELEMS_PER_THREAD_ASYNC

    # GMEM → SMEM via cp.async (bypasses registers!)
    # Issue one 32-bit async copy per element
    for i in range(ELEMS_PER_THREAD_ASYNC):
        gmem_elem = cute.make_tensor(
            gIn.iterator + base_idx + i,
            cute.make_layout((1, 1), stride=(0, 1))
        )
        smem_elem = cute.make_tensor(
            smem_ptr + smem_offset + i,
            cute.make_layout((1, 1), stride=(0, 1))
        )
        cute.copy(cp_async_atom, gmem_elem, smem_elem)

    # Commit and wait for async copy to complete
    cute.arch.cp_async_commit_group()
    cute.arch.cp_async_wait_group(0)
    cute.arch.sync_threads()

    # SMEM → RMEM: load into registers for computation
    rmem = cute.make_rmem_tensor(ELEMS_PER_THREAD_ASYNC, cutlass.Float32)
    for i in range(ELEMS_PER_THREAD_ASYNC):
        rmem[i] = smem[smem_offset + i]

    # Compute in registers
    for i in range(ELEMS_PER_THREAD_ASYNC):
        rmem[i] = rmem[i] * 2.0

    # RMEM → GMEM
    for i in range(ELEMS_PER_THREAD_ASYNC):
        gOut[base_idx + i] = rmem[i]


@cute.jit
def cp_async_demo(mIn: cute.Tensor, mOut: cute.Tensor):
    N = mIn.shape[0]
    cp_async_kernel(mIn, mOut).launch(
        grid=(N // SMEM_SIZE, 1, 1),
        block=(BLOCK_SIZE_ASYNC, 1, 1),
    )


# Test correctness
N = 1 << 20
inp = torch.randn(N, device="cuda", dtype=torch.float32)
out = torch.zeros(N, device="cuda", dtype=torch.float32)

inp_ = from_dlpack(inp, assumed_align=16)
out_ = from_dlpack(out, assumed_align=16)

cp_async_fn = cute.compile(cp_async_demo, inp_, out_)
cp_async_fn(inp_, out_)

torch.testing.assert_close(out, inp * 2.0)
print(f"GMEM → SMEM (cp.async) → RMEM → GMEM: PASSED (N={N:,})")
print(f"  Input sample:  {inp[:4].tolist()}")
print(f"  Output sample: {out[:4].tolist()}")

GMEM → SMEM (cp.async) → RMEM → GMEM: PASSED (N=1,048,576)
  Input sample:  [-0.21047256886959076, -0.22813226282596588, 0.06138579919934273, 0.9345427751541138]
  Output sample: [-0.4209451377391815, -0.45626452565193176, 0.12277159839868546, 1.8690855503082275]


### cp.async vs naive GMEM → SMEM

The difference between the naive path and `cp.async`:

```
Naive:     GMEM  →  Registers  →  SMEM     (2 hops, uses registers as temporary)
cp.async:  GMEM  →  SMEM                   (1 hop, direct DMA by the memory subsystem)
```

**`cp.async` advantages:**
    - **Frees up registers** data doesn't occupy register slots during transit
    - **Asynchronous** -- the thread issues the copy and continues doing other work
- **Higher throughput** -- the memory controller can pipeline multiple 128-bit async copies

**The async handshake:**
1. `cute.copy_atom_call(cp_async_atom, src, dst)`: initiates the async copy
2. `cute.arch.cp_async_commit_group()`: commits all pending async copies into a "group"
3. `cute.arch.cp_async_wait_group(N)`: waits until at most N groups are still in-flight (0 = wait for all)
4. `cute.arch.sync_threads()`: ensures all threads have completed the wait before reading SMEM

In [16]:
import re

def count_registers(ptx: str) -> dict:
    """Parse .reg declarations from PTX and count total registers by type."""
    counts = {}
    for match in re.finditer(r'\.reg\s+\.(\w+)\s+%\w+<(\d+)>', ptx):
        reg_type, count = match.group(1), int(match.group(2))
        counts[reg_type] = counts.get(reg_type, 0) + count
    return counts

naive_regs = count_registers(gmem_smem_rmem_fn.__ptx__)
async_regs = count_registers(cp_async_fn.__ptx__)

print('Register allocation comparison (both kernels: 4 elements/thread):')
print(f"{'Type':<8} {'Path 2 (naive)':>16} {'Path 3 (cp.async)':>18}")
print('-' * 44)
all_types = sorted(set(naive_regs) | set(async_regs))
for t in all_types:
    n = naive_regs.get(t, 0)
    a = async_regs.get(t, 0)
    diff = a - n
    sign = '+' if diff > 0 else ''
    print(f'.{t:<7} {n:>16} {a:>18}  ({sign}{diff})')
print()
total_n = sum(naive_regs.values())
total_a = sum(async_regs.values())
print(f"{'Total':<8} {total_n:>16} {total_a:>18}  ({'+' if total_a-total_n > 0 else ''}{total_a-total_n})")

Register allocation comparison (both kernels: 4 elements/thread):
Type       Path 2 (naive)  Path 3 (cp.async)
--------------------------------------------
.b32                    9                 12  (+3)
.b64                    6                  9  (+3)
.f32                   13                  9  (-4)

Total                  28                 30  (+2)


## Using `autovec_copy` for Automatic Vectorization

CuTe DSL provides `autovec_copy` -- a copy function that analyzes the pointer alignment and layout of source and destination tensors to automatically select the widest safe vector width. Under the hood, it creates a `CopyUniversalOp` atom with the optimal `num_bits_per_copy`.

This is the simplest way to get vectorized copies between any memory spaces:

```python
cute.autovec_copy(src_tensor, dst_tensor)
```

`autovec_copy` examines:
1. The **layout alignment** (stride patterns) of both tensors
2. The **pointer alignment** (byte alignment of the base address)
3. Caps at 256 bits maximum

Let's use it to copy between GMEM and RMEM with automatic vectorization.

In [17]:
ELEMS_PER_THREAD_VEC = 4

@cute.kernel
def autovec_kernel(gIn: cute.Tensor, gOut: cute.Tensor):
    """Uses autovec_copy for automatic vectorization of GMEM↔RMEM copies."""
    tidx, _, _ = cute.arch.thread_idx()
    bidx, _, _ = cute.arch.block_idx()
    bdim, _, _ = cute.arch.block_dim()
    global_tid = bidx * bdim + tidx
    base_idx = global_tid * ELEMS_PER_THREAD_VEC

    # Create a sub-tensor for this thread's GMEM slice
    # Using a 2D layout so autovec_copy can reason about vector width
    vec_layout = cute.make_layout((ELEMS_PER_THREAD_VEC, 1), stride=(1, 0))
    src = cute.make_tensor(gIn.iterator + base_idx, vec_layout)

    # Register-backed tensor with matching shape
    rmem = cute.make_rmem_tensor((ELEMS_PER_THREAD_VEC, 1), cutlass.Float32)

    # GMEM → RMEM: autovec_copy picks the widest safe vector width
    cute.autovec_copy(src, rmem)

    # Compute in registers
    for i in range(ELEMS_PER_THREAD_VEC):
        rmem[i, 0] = rmem[i, 0] * 2.0

    # RMEM → GMEM
    dst = cute.make_tensor(gOut.iterator + base_idx, vec_layout)
    cute.autovec_copy(rmem, dst)


@cute.jit
def autovec_demo(mIn: cute.Tensor, mOut: cute.Tensor):
    N = mIn.shape[0]
    threads_per_block = 256
    blocks = N // (threads_per_block * ELEMS_PER_THREAD_VEC)
    autovec_kernel(mIn, mOut).launch(
        grid=(blocks, 1, 1),
        block=(threads_per_block, 1, 1),
    )


# Test correctness
N = 1 << 20
inp = torch.randn(N, device="cuda", dtype=torch.float32)
out = torch.zeros(N, device="cuda", dtype=torch.float32)

inp_ = from_dlpack(inp, assumed_align=16)
out_ = from_dlpack(out, assumed_align=16)

autovec_fn = cute.compile(autovec_demo, inp_, out_)
autovec_fn(inp_, out_)

torch.testing.assert_close(out, inp * 2.0)
print(f"autovec_copy GMEM → RMEM → GMEM: PASSED (N={N:,})")

autovec_copy GMEM → RMEM → GMEM: PASSED (N=1,048,576)


### Why vectorization matters

Under the hood, `autovec_copy` selects the widest load/store instruction that the pointer alignment allows. Wider vector loads issue fewer instructions for the same data:

| Vector Width | PTX Instruction | Bytes/instruction | FP32 Elements | FP16/BF16 Elements |
|-------------|----------------|-------------------|---------------|---------------------|
| 32-bit | `LD.GLOBAL.B32` | 4 | 1 | 2 |
| 64-bit | `LD.GLOBAL.B64` | 8 | 2 | 4 |
| 128-bit | `LD.GLOBAL.B128` | 16 | 4 | 8 |

Fewer instructions means less scheduling overhead and better memory bus utilization. The `autovec_copy` function handles this automatically based on the tensor's alignment properties.

## Why CopyAtoms? From Scalar Loads to Hardware-Mapped Instructions

The manual copy loop in Path 1 and even `autovec_copy` are both **general-purpose** — they work anywhere but give you limited control over which hardware instruction is actually emitted. CopyAtoms give you **explicit, hardware-mapped control**: you pick the instruction, the width, and the memory path. The compiler doesn't guess.

### The three levels of copy abstraction

| Approach | Control | Instruction |
|----------|---------|-------------|
| Manual loop (`rmem[i] = gIn[...]`) | None — compiler decides | Scalar `LD.GLOBAL` per element |
| `autovec_copy` | Automatic — inferred from alignment | Widest *safe* vector load |
| `CopyAtom` | Explicit — you specify the op and width | Exactly the instruction you name |

### What the manual loop actually generates

The loop in `gmem_to_rmem_kernel` produces **one load per element** — four separate 32-bit loads for four floats:

```ptx
ld.global.f32  %f1, [%rd4];
ld.global.f32  %f2, [%rd4+4];
ld.global.f32  %f3, [%rd4+8];
ld.global.f32  %f4, [%rd4+12];
```

A `CopyUniversalOp` atom with `num_bits_per_copy=128` collapses these into a single **vectorized load**:

```ptx
ld.global.v4.f32  {%f1, %f2, %f3, %f4}, [%rd4];
```

### Pointer arithmetic kills alignment annotations

`autovec_copy` is described as selecting *the widest safe width based on pointer alignment*. "Safe" is key: if the compiler can only prove 32-bit alignment, it falls back to scalar loads silently. An explicit CopyAtom at 128-bit **fails loudly** if alignment isn't provably 128 bits.

The catch: raw pointer arithmetic (`gIn.iterator + offset`) drops the alignment annotation from `assumed_align=16` down to the natural element alignment (32 bits for fp32). **Both `autovec_copy` and an explicit atom will therefore use scalar loads when the sub-tensor is created this way.**

The fix is `cute.local_tile`: it encodes the per-thread offset in the *layout* rather than adding to the pointer, keeping the base address and letting CuTe derive alignment from the stride:

```
stride alignment = 4 fp32 × 32 bits = 128 bits  ✓  (matches assumed_align=16)
```

### Why go beyond `autovec_copy`?

With proper alignment, `autovec_copy` and `CopyUniversalOp(128)` produce identical PTX. The value of explicit CopyAtoms is **unlocking hardware paths that `autovec_copy` cannot express**:

| CopyAtom | What it unlocks |
|----------|-----------------|
| `CopyUniversalOp` (explicit) | Fails loudly on bad alignment; composable with CuTe tiling |
| `cpasync.CopyG2SOp` | Async GMEM→SMEM that bypasses registers entirely (Path 3 above) |
| `warp.LdMatrix` | Loads 8×8 matrix tiles in the swizzled layout tensor cores expect |
| `cpasync.CopyBulkTensorTile*` | TMA: hardware-managed multi-dimensional tiled copies |

Let's see the instruction-level difference and then benchmark it.

In [18]:
ELEMS_PER_THREAD_ATOM = 4  # 128 bits / 32 bits per fp32 = 4 elements

@cute.kernel
def copyatom_kernel(gIn: cute.Tensor, gOut: cute.Tensor):
    """Explicit CopyUniversalOp at 128-bit width.

    Key: uses local_tile instead of (gIn.iterator + offset).
    Raw pointer arithmetic loses the alignment annotation from assumed_align=16;
    local_tile encodes the per-thread offset in the layout while keeping the
    original base pointer, so CuTe can verify alignment statically:
      stride alignment = 4 fp32 * 32 bits = 128 bits  ✓
    """
    tidx, _, _ = cute.arch.thread_idx()
    bidx, _, _ = cute.arch.block_idx()
    bdim, _, _ = cute.arch.block_dim()
    global_tid = bidx * bdim + tidx

    # Extract this thread's (4,) tile via layout arithmetic, not pointer arithmetic
    src  = cute.local_tile(gIn,  (ELEMS_PER_THREAD_ATOM,), (global_tid,))
    rmem = cute.make_rmem_tensor((ELEMS_PER_THREAD_ATOM,), cutlass.Float32)

    # Explicit 128-bit CopyAtom — requires 128-bit aligned src, satisfied above
    atom = cute.make_copy_atom(cute.nvgpu.CopyUniversalOp(), cutlass.Float32, num_bits_per_copy=128)

    # GMEM -> RMEM: one LD.GLOBAL.V4.F32 instruction
    cute.copy(atom, src, rmem)

    # Compute in registers
    for i in range(ELEMS_PER_THREAD_ATOM):
        rmem[i] = rmem[i] * 2.0

    # RMEM -> GMEM: one ST.GLOBAL.V4.F32 instruction
    dst = cute.local_tile(gOut, (ELEMS_PER_THREAD_ATOM,), (global_tid,))
    cute.copy(atom, rmem, dst)


@cute.jit
def copyatom_demo(mIn: cute.Tensor, mOut: cute.Tensor):
    N = mIn.shape[0]
    threads_per_block = 256
    blocks = N // (threads_per_block * ELEMS_PER_THREAD_ATOM)
    copyatom_kernel(mIn, mOut).launch(
        grid=(blocks, 1, 1),
        block=(threads_per_block, 1, 1),
    )


N = 1 << 20
inp = torch.randn(N, device="cuda", dtype=torch.float32)
out = torch.zeros(N, device="cuda", dtype=torch.float32)
inp_ = from_dlpack(inp, assumed_align=16)
out_ = from_dlpack(out, assumed_align=16)

copyatom_fn = cute.compile(copyatom_demo, inp_, out_)
copyatom_fn(inp_, out_)
torch.testing.assert_close(out, inp * 2.0)
print(f"CopyAtom GMEM -> RMEM -> GMEM: PASSED (N={N:,})")

# --- PTX comparison: manual vs autovec_copy vs explicit CopyAtom ---
import re

def count_ld_st(ptx: str) -> dict:
    """Count LD/ST instructions by full qualifier (handles v4.f32, b128, f32 forms)."""
    counts = {}
    for m in re.finditer(r'\b((?:ld|st)\.global(?:\.\w+)+)', ptx, re.IGNORECASE):
        key = m.group(1).lower()
        counts[key] = counts.get(key, 0) + 1
    return counts

manual_lds  = count_ld_st(gmem_to_rmem_fn.__ptx__)
autovec_lds = count_ld_st(autovec_fn.__ptx__)
atom_lds    = count_ld_st(copyatom_fn.__ptx__)

print("\nPTX load/store instruction count (4 fp32 elements per thread):")
print(f"{'Instruction':<28} {'Manual':>10} {'autovec':>10} {'CopyAtom':>10}")
print("-" * 60)
for k in sorted(set(manual_lds) | set(autovec_lds) | set(atom_lds)):
    print(f"{k:<28} {manual_lds.get(k, 0):>10} {autovec_lds.get(k, 0):>10} {atom_lds.get(k, 0):>10}")

# Print the actual LD/ST lines from each PTX
for label, ptx in [("Manual", gmem_to_rmem_fn.__ptx__),
                   ("autovec_copy", autovec_fn.__ptx__),
                   ("CopyAtom", copyatom_fn.__ptx__)]:
    ld_lines = [l.strip() for l in ptx.splitlines() if re.search(r'\b(?:ld|st)\.global', l, re.I)]
    print(f"\n--- {label}: LD/ST PTX lines ---")
    for l in ld_lines:
        print(" ", l)


CopyAtom GMEM -> RMEM -> GMEM: PASSED (N=1,048,576)

PTX load/store instruction count (4 fp32 elements per thread):
Instruction                      Manual    autovec   CopyAtom
------------------------------------------------------------
ld.global.f32                         4          4          0
ld.global.v4.f32                      0          0          1
st.global.f32                         4          4          0
st.global.v4.f32                      0          0          1

--- Manual: LD/ST PTX lines ---
  ld.global.f32 	%f1, [%rd4];
  ld.global.f32 	%f2, [%rd4+4];
  ld.global.f32 	%f3, [%rd4+8];
  ld.global.f32 	%f4, [%rd4+12];
  st.global.f32 	[%rd5], %f5;
  st.global.f32 	[%rd5+4], %f6;
  st.global.f32 	[%rd5+8], %f7;
  st.global.f32 	[%rd5+12], %f8;

--- autovec_copy: LD/ST PTX lines ---
  ld.global.f32 	%f1, [%rd4];
  ld.global.f32 	%f2, [%rd4+4];
  ld.global.f32 	%f3, [%rd4+8];
  ld.global.f32 	%f4, [%rd4+12];
  st.global.f32 	[%rd5], %f5;
  st.global.f32 	[%rd5+4], %f6

### What the PTX tells us

The manual loop emits one load per element — `ld.global.f32` × 4. The CopyAtom and `autovec_copy` emit a single vectorized load — `ld.global.v4.f32` × 1 (four fp32 values in one instruction).

> **Note:** The exact opcode is arch-dependent. The PTX lines printed above show what was actually emitted on this machine.

**Why fewer, wider instructions help — and when they don't:**

For a simple, perfectly-coalesced 1D copy like this one, the bandwidth benchmark shows roughly the same throughput for scalar and vector loads. This is because **hardware coalescing** already handles it: the 32 threads in a warp issue their 32 individual `ld.global.f32` requests to consecutive addresses, and the L2 cache merges them into one 128-byte transaction. The scalar and vector PTX paths produce the same number of memory transactions.

Where the instruction count difference *does* matter:

- **Compute-heavy kernels**: the warp scheduler has a fixed number of issue slots per cycle. Fewer memory instructions leave more slots for arithmetic, improving overlap between compute and memory.
- **Complex access patterns**: non-unit strides or gather/scatter patterns may not coalesce cleanly; wider vector loads can help.
- **Register pressure**: fewer instructions per thread → smaller instruction footprint → better occupancy.

### The real payoff: special hardware paths

`CopyUniversalOp` maps to the standard `LD/ST.GLOBAL` family and shows the width advantage clearly in context. The deeper value of explicit CopyAtoms is unlocking paths that **cannot be expressed any other way**:

- **`cp.async`** (Path 3 above): data moves GMEM→SMEM without touching registers. The thread issues the copy and keeps executing — memory latency is hidden rather than blocking the warp.
- **`ldmatrix`**: loads 8×8 matrix tiles in exactly the swizzled layout tensor cores expect. Scalar loads would need complex index arithmetic the compiler cannot auto-vectorize.
- **TMA**: the Tensor Memory Accelerator (Hopper+) fills shared memory tiles autonomously. No per-element thread work at all.

Each of these is only accessible via a CopyAtom.

In [20]:
# Bandwidth benchmark: manual scalar copy vs explicit 128-bit CopyAtom
# On a simple coalesced copy the hardware fuses scalar warp loads into 128-byte
# transactions anyway -- expect similar bandwidth, but fewer PTX instructions
# with the CopyAtom (less scheduler pressure in compute-heavy kernels).
N_BENCH = 1 << 24  # 16M elements (~64 MB fp32)
inp_b = torch.randn(N_BENCH, device="cuda", dtype=torch.float32)
out_b = torch.zeros(N_BENCH, device="cuda", dtype=torch.float32)
inp_b_ = from_dlpack(inp_b, assumed_align=16)
out_b_ = from_dlpack(out_b, assumed_align=16)

manual_fn_b = cute.compile(gmem_to_rmem,  inp_b_, out_b_)
atom_fn_b   = cute.compile(copyatom_demo, inp_b_, out_b_)

t_manual = triton.testing.do_bench(lambda: manual_fn_b(inp_b_, out_b_), warmup=25, rep=100)
t_atom   = triton.testing.do_bench(lambda: atom_fn_b(inp_b_, out_b_),   warmup=25, rep=100)

bytes_moved = N_BENCH * 4 * 2  # read + write, 4 bytes/fp32
gb_manual = bytes_moved / t_manual / 1e6
gb_atom   = bytes_moved / t_atom   / 1e6

print(f"Memory bandwidth benchmark  (N={N_BENCH:,}, {bytes_moved/1e9:.1f} GB moved per pass)")
print(f"  Manual scalar copy:  {t_manual:6.2f} ms  ->  {gb_manual:6.1f} GB/s")
print(f"  CopyAtom 128-bit:    {t_atom:6.2f} ms  ->  {gb_atom:6.1f} GB/s")
print(f"  Speedup:             {t_manual/t_atom:.2f}x")
print()
print("Both kernels saturate the same memory bandwidth: the GPU's L2 coalesces the")
print("32 scalar warp loads into one 128-byte transaction regardless of PTX width.")
print("The CopyAtom advantage on GMEM<->RMEM is fewer instruction issue slots, not")
print("extra memory transactions -- and it unlocks special paths (cp.async, ldmatrix,")
print("TMA) that have no scalar equivalent.")


Memory bandwidth benchmark  (N=16,777,216, 0.1 GB moved per pass)
  Manual scalar copy:    0.35 ms  ->   380.7 GB/s
  CopyAtom 128-bit:      0.35 ms  ->   382.1 GB/s
  Speedup:             1.00x


## CuTe DSL Copy Atom Reference

CuTe DSL provides a rich set of **CopyAtom** types, each mapping to a specific hardware copy instruction. Below is a comprehensive catalog of every copy operation available in the `cutlass.cute` package.

### How Copy Atoms Work

A CopyAtom wraps a hardware instruction and defines:
- **The source and destination memory spaces** (GMEM, SMEM, RMEM)
- **The data layout** each thread expects (how many elements, what pattern)
- **The vector width** (how many bits per copy instruction)

Usage pattern:
```python
# 1. Create a CopyOp (describes the hardware instruction)
op = cute.nvgpu.CopyUniversalOp()

# 2. Create a CopyAtom from the op (adds type and width info)
atom = cute.make_copy_atom(op, cutlass.Float32, num_bits_per_copy=128)

# 3. Execute the copy on tensors with matching layout profile (V,)
cute.copy_atom_call(atom, src_tensor, dst_tensor)
```

### Copy Functions

| Function | Description |
|----------|-------------|
| `cute.copy(atom, src, dst)` | Main copy algorithm for tensors with layout profile `(V, Rest...)`. Handles recursive decomposition of multi-mode tensors. Supports predication via `pred=` kwarg. |
| `cute.copy_atom_call(atom, src, dst)` | Low-level single-atom copy for tensors with layout profile `(V,)`. No recursion -- directly issues one copy instruction. |
| `cute.basic_copy(src, dst)` | Simple element-wise copy. No atom needed -- uses SIMT sync copy internally. |
| `cute.basic_copy_if(pred, src, dst)` | Predicated element-wise copy. Copies `src[i]` to `dst[i]` only where `pred[i]` is true. |
| `cute.autovec_copy(src, dst)` | Auto-vectorizing copy. Analyzes layout alignment to pick the widest safe vector width automatically. |
| `cute.prefetch(atom, src)` | Prefetches data from GMEM into L2 cache. Currently only supports TMA prefetch atoms. |

### 1. Universal Copy, `cute.nvgpu.CopyUniversalOp`

**Path:** Any → Any (GMEM↔RMEM, SMEM↔RMEM, GMEM↔SMEM via registers)
**Architecture:** All (SM50+)
**PTX:** `LD.GLOBAL`, `LD.SHARED`, `ST.GLOBAL`, `ST.SHARED`, etc.

The general-purpose copy. Maps to standard load/store instructions. Supports vectorization up to 256 bits.

```python
op = cute.nvgpu.CopyUniversalOp()
atom = cute.make_copy_atom(
    op,
    cutlass.Float32,             # element type (determines layout)
    num_bits_per_copy=128,       # vectorization: 32, 64, 128, or 256 bits (0 = auto)
    l1c_evict_priority=cute.nvgpu.CacheEvictionPriority.EVICT_NORMAL,  # L1 cache hint
    memory_order=cute.nvgpu.MemoryOrder.WEAK,          # memory ordering
    memory_scope=cute.nvgpu.MemoryScope.CTA,           # visibility scope
    invariant=False,             # True = use read-only cache (LDG)
)
```

**Parameters:**
| Parameter | Options | Description |
|-----------|---------|-------------|
| `num_bits_per_copy` | 0, 32, 64, 128, 256 | Bits per copy instruction. 0 = auto-vectorize |
| `l1c_evict_priority` | `EVICT_NORMAL`, `EVICT_FIRST`, `EVICT_LAST`, `EVICT_UNCHANGED`, `NO_ALLOCATE` | L1 cache eviction hint |
| `memory_order` | `WEAK`, `RELAXED`, `ACQUIRE`, `RELEASE`, `ACQ_REL`, `SC`, `MMIO`, `CONSTANT`, `VOLATILE` | Memory ordering semantics |
| `memory_scope` | `CTA`, `CLUSTER`, `GPU`, `SYS` | Visibility scope for ordering |
| `invariant` | `True`/`False` | Read-only optimization (texture cache path) |

### 2. Asynchronous GMEM → SMEM, `cute.nvgpu.cpasync.CopyG2SOp`

**Path:** GMEM → SMEM (bypasses registers)
**Architecture:** SM80+ (Ampere)
**PTX:** `cp.async`

Initiates an asynchronous copy from global memory directly to shared memory. The thread does not wait for completion -- you must explicitly commit and wait.

```python
op = cute.nvgpu.cpasync.CopyG2SOp(
    cache_mode=cute.nvgpu.cpasync.LoadCacheMode.ALWAYS  # L1 caching policy
)
atom = cute.make_copy_atom(op, cutlass.Float32, num_bits_per_copy=128)
```

**Cache modes:**
| Mode | Description |
|------|-------------|
| `ALWAYS` | Cache in L1 (default) |
| `GLOBAL` | Cache in L2 only, bypass L1 |
| `STREAMING` | Streaming access, evict first |
| `LAST_USE` | Hint that this is the last access |
| `NONE` | No caching |

**Synchronization protocol:**
```python
cute.copy_atom_call(cp_async_atom, gmem_src, smem_dst)  # initiate
cute.arch.cp_async_commit_group()                         # commit pending copies
cute.arch.cp_async_wait_group(0)                          # wait for all groups
cute.arch.sync_threads()                                  # block-wide barrier
```

### 3. TMA Bulk Tensor Copy, `cute.nvgpu.cpasync.CopyBulkTensorTile*`

**Architecture:** SM90+ (Hopper)
**PTX:** `cp.async.bulk.tensor`

The **Tensor Memory Accelerator (TMA)** is a dedicated hardware unit on Hopper+ GPUs that can copy entire multi-dimensional tiles between GMEM and SMEM. Unlike `cp.async` (which is thread-initiated), TMA operations are issued by a single thread and the hardware handles the entire tile transfer.

| CopyOp | Path | Description |
|--------|------|-------------|
| `CopyBulkTensorTileG2SOp` | GMEM → SMEM | Bulk tensor tile load |
| `CopyBulkTensorTileG2SMulticastOp` | GMEM → SMEM (multicast) | Load + broadcast to multiple CTAs in a cluster |
| `CopyBulkTensorTileS2GOp` | SMEM → GMEM | Bulk tensor tile store |
| `CopyReduceBulkTensorTileS2GOp` | SMEM → GMEM (reduce) | Store with atomic reduction (ADD, MIN, MAX, etc.) |

```python
# GMEM → SMEM via TMA
op = cute.nvgpu.cpasync.CopyBulkTensorTileG2SOp(
    cta_group=cute.nvgpu.tcgen05.CtaGroup.ONE  # ONE or TWO (2-CTA cooperative)
)

# TMA requires a tensor map descriptor, built via make_tiled_tma_atom
tma_atom, tma_tensor = cute.nvgpu.cpasync.make_tiled_tma_atom(
    op, gmem_tensor, smem_layout, cta_tiler
)

# Execute with mbarrier synchronization
cute.copy(tma_atom, src, dst, tma_bar_ptr=mbar_ptr, mcast_mask=mask)
```

**TMA key features:**
- Single-thread issue: only one thread needs to initiate the copy
- Hardware handles address computation for multi-dimensional tiles
- Supports multicast to multiple CTAs in a cluster (SM90+)
- Uses mbarrier for synchronization instead of cp.async groups

### 4. Bulk Copy (Non-Tensor), `CopyBulk*Op`

**Architecture:** SM90+ (Hopper)
**PTX:** `cp.async.bulk`

Lower-level bulk copy operations that work with raw addresses (not tensor descriptors). These are used internally by CUTLASS but are not part of the public API (`__all__`).

| CopyOp | Path | Description |
|--------|------|-------------|
| `CopyBulkG2SOp` | GMEM → SMEM | Bulk raw copy |
| `CopyBulkG2SMulticastOp` | GMEM → SMEM | Bulk raw copy with multicast |
| `CopyBulkS2GOp` | SMEM → GMEM | Bulk raw store |
| `CopyBulkS2GByteMaskOp` | SMEM → GMEM | Bulk store with per-byte mask (SM100+) |
| `CopyBulkS2SOp` | SMEM → SMEM (cross-CTA) | Copy between CTAs in a cluster |
| `CopyDsmemStoreOp` | RMEM → DSMEM | Async store to distributed shared memory |

### 5. Warp-Level Matrix Load/Store, `cute.nvgpu.warp.LdMatrix*` / `StMatrix*`

**Path:** SMEM ↔ RMEM (structured for Tensor Core consumption)
**Architecture:** SM75+ (Turing)
**PTX:** `ldmatrix`, `stmatrix`

These are **warp-level** instructions: all 32 threads in a warp cooperate to load/store a matrix tile from SMEM into registers in the exact layout that Tensor Cores expect. This avoids expensive register shuffles before MMA instructions.

#### Load Matrix (SMEM → RMEM)

| CopyOp | Matrix Shape | Element Size | Notes |
|--------|-------------|-------------|-------|
| `LdMatrix8x8x16bOp` | 8x8 | 16-bit | Basic matrix load, `.m8n8` qualifier |
| `LdMatrix8x16x8bOp` | 8x16 | 8-bit | Supports 4-bit and 6-bit unpacking |
| `LdMatrix16x8x8bOp` | 16x8 | 8-bit | Transpose required, lowers to `.m16n16` + permutation |
| `LdMatrix16x16x8bOp` | 16x16 | 8-bit | Transpose + optional unpacking (4b, 6b) |

```python
op = cute.nvgpu.warp.LdMatrix8x8x16bOp(
    transpose=False,     # whether to transpose the loaded matrix
    num_matrices=4,      # how many matrices to load (1, 2, or 4)
)
atom = cute.make_copy_atom(op, cutlass.Float16)
```

#### Store Matrix (RMEM → SMEM)

| CopyOp | Matrix Shape | Element Size | Notes |
|--------|-------------|-------------|-------|
| `StMatrix8x8x16bOp` | 8x8 | 16-bit | Basic matrix store |
| `StMatrix16x8x8bOp` | 16x8 | 8-bit | Transpose store |

```python
op = cute.nvgpu.warp.StMatrix8x8x16bOp(
    transpose=False,
    num_matrices=4,
)
atom = cute.make_copy_atom(op, cutlass.Float16)
```

**When to use:** Before/after MMA (matrix multiply-accumulate) operations. The `ldmatrix`/`stmatrix` instructions are designed to load data from SMEM into registers in exactly the layout that `mma.sync` instructions expect, avoiding costly register shuffles.

### Summary: Which CopyAtom for Which Path?

| Path | Best CopyAtom | Architecture | Use Case |
|------|--------------|-------------|----------|
| **GMEM → RMEM** | `CopyUniversalOp` | All | Element-wise ops, simple loads |
| **RMEM → GMEM** | `CopyUniversalOp` | All | Writing results back |
| **GMEM → SMEM** | `CopyG2SOp` (cp.async) | SM80+ | Staging data for block-wide reuse |
| **GMEM → SMEM** | `CopyBulkTensorTileG2SOp` (TMA) | SM90+ | Large tile loads, GEMM |
| **SMEM → GMEM** | `CopyBulkTensorTileS2GOp` (TMA) | SM90+ | Tile stores |
| **SMEM → RMEM** | `CopyUniversalOp` | All | General SMEM reads |
| **SMEM → RMEM** | `LdMatrix*Op` (ldmatrix) | SM75+ | Loading MMA operands |
| **RMEM → SMEM** | `CopyUniversalOp` | All | General SMEM writes |
| **RMEM → SMEM** | `StMatrix*Op` (stmatrix) | SM75+ | Storing MMA results |

The evolution across GPU generations is clear:
- **Turing (SM75):** Introduced `ldmatrix`/`stmatrix` for structured SMEM↔RMEM transfers
- **Ampere (SM80):** Added `cp.async` for register-free GMEM→SMEM
- **Hopper (SM90):** Added TMA for hardware-managed multi-dimensional tile transfers


## What's Next: Tensor Memory (TMEM)

Blackwell (SM100+) introduces a fifth memory tier: **Tensor Memory (TMEM)**. Unlike the four tiers covered here, TMEM is not a general-purpose scratchpad — it is a dedicated register file for the Tensor Core pipeline, only accessible via the `tcgen05` instruction family (`cp.async` SMEM→TMEM, and `ld`/`st` TMEM↔RMEM).

TMEM and the Blackwell Tensor Core programming model will be covered in a future notebook.